# OMI Transactions Exploration

Exploratory analysis of OMI real estate transaction volumes, with a focus on the Number of Normalised Transactions (NTN) and its geographical and dimensional distribution.

## 1. Setup

In [84]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import re

In [85]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# CONFIGURATION
# =============================================================================

CWD = Path.cwd()

PROJECT_ROOT = CWD.parent

TRANSACTIONS_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "transactions"
)

## 2. Import Data

The OMI transaction data are provided through multiple files for each reference year.

The available datasets include:

- `lista-com`: municipality reference data
- `valori-com`: transaction volumes by municipality
- `valori-per`: transaction volumes at provincial level
- `valori-res`: residential market statistics

In [86]:
YEAR_FOLDERS = sorted(
    folder
    for folder in TRANSACTIONS_FOLDER.iterdir()
    if folder.is_dir() and folder.name.isdigit()
)

for folder in YEAR_FOLDERS:
    print(folder.name)

2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025


In [87]:
def get_transaction_files(year_folder: Path) -> dict[str, Path]:
    """Return the four OMI transaction files for a given year."""

    files = {
        file.name.lower(): file
        for file in year_folder.iterdir()
        if file.is_file()
    }

    return {
        "lista_com": next(
            file
            for name, file in files.items()
            if "lista-com" in name
        ),
        "valori_com": next(
            file
            for name, file in files.items()
            if "valori-com" in name
        ),
        "valori_per": next(
            file
            for name, file in files.items()
            if "valori-per" in name
        ),
        "valori_res": next(
            file
            for name, file in files.items()
            if "valori-res" in name
        ),
    }

for year_folder in YEAR_FOLDERS:
    print(f"\n{year_folder.name}")

    transaction_files = get_transaction_files(year_folder)

    for dataset, file in transaction_files.items():
        print(f"  {dataset}: {file.name}")


2011
  lista_com: 2011_LISTA-COM.csv
  valori_com: 2011_VALORI-COM.csv
  valori_per: 2011_VALORI-PER.csv
  valori_res: 2011_VALORI-RES.csv

2012
  lista_com: 2012_LISTA-COM.csv
  valori_com: 2012_VALORI-COM.csv
  valori_per: 2012_VALORI-PER.csv
  valori_res: 2012_VALORI-RES.csv

2013
  lista_com: 2013_LISTA-COM.csv
  valori_com: 2013_VALORI-COM.csv
  valori_per: 2013_VALORI-PER.csv
  valori_res: 2013_VALORI-RES.csv

2014
  lista_com: 2014_LISTA-COM.csv
  valori_com: 2014_VALORI-COM.csv
  valori_per: 2014_VALORI-PER.csv
  valori_res: 2014_VALORI-RES.csv

2015
  lista_com: 2015_LISTA-COM.csv
  valori_com: 2015_VALORI-COM.csv
  valori_per: 2015_VALORI-PER.csv
  valori_res: 2015_VALORI-RES.csv

2016
  lista_com: 2016_LISTA-COM.csv
  valori_com: 2016_VALORI-COM.csv
  valori_per: 2016_VALORI-PER.csv
  valori_res: 2016_VALORI-RES.csv

2017
  lista_com: 2017_LISTA-COM.csv
  valori_com: 2017_VALORI-COM.csv
  valori_per: 2017_VALORI-PER.csv
  valori_res: 2017_VALORI-RES.csv

2018
  lista_com: 2

In [88]:
lista_com = {}
valori_com = {}
valori_per = {}
valori_res = {}

for year_folder in YEAR_FOLDERS:

    year = int(year_folder.name)

    files = get_transaction_files(year_folder)

    lista_com[year] = pd.read_csv(
        files["lista_com"],
        sep=";",
        decimal=","
    )

    valori_com[year] = pd.read_csv(
        files["valori_com"],
        sep=";",
        decimal=","
    )

    valori_per[year] = pd.read_csv(
        files["valori_per"],
        sep=";",
        decimal=","
    )

    valori_res[year] = pd.read_csv(
        files["valori_res"],
        sep=";",
        decimal=","
    )

In [89]:
lista_com[2025].head()

,Area,Regione,Provincia,2025_CodCom,Comune,Cap,TAGLIA MERCATO
0,Nord Est,Veneto,PD,A001,ABANO TERME,NonCap,M
1,Nord Ovest,Lombardia,LO,A004,ABBADIA CERRETO,NonCap,S
2,Nord Ovest,Lombardia,LC,A005,ABBADIA LARIANA,NonCap,S
3,Centro,Toscana,SI,A006,ABBADIA SAN SALVATORE,NonCap,S
4,Isole,Sardegna,OR,A007,ABBASANTA,NonCap,S


In [90]:
valori_res[2018].head()

,AREA,Regione,prov,2018_CodCom,NTN fino a 50 mq,NTN 50 -| 85 mq,NTN 85 -| 115 mq,NTN 115 -| 145 mq,NTN oltre 145 mq,NTN 2018
0,Nord Est,Veneto,PD,A001,14.0,70.25,69.00,50.10,88.14,291.49
1,Nord Ovest,Lombardia,LO,A004,0.0,0.00,0.00,0.00,1.00,1.00
2,Nord Ovest,Lombardia,LC,A005,4.0,11.50,7.50,8.96,6.00,37.96
3,Centro,Toscana,SI,A006,6.0,26.75,24.75,24.50,9.25,91.25
4,Isole,Sardegna,OR,A007,0.0,4.00,1.00,0.00,7.00,12.00


In [91]:
print("lista-com:", sorted(lista_com))
print("valori-com:", sorted(valori_com))
print("valori-per:", sorted(valori_per))
print("valori-res:", sorted(valori_res))

lista-com: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
valori-com: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
valori-per: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
valori-res: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [92]:
assert set(lista_com) == set(valori_com)
assert set(lista_com) == set(valori_per)
assert set(lista_com) == set(valori_res)

## 3. Dataset Processing

In [93]:
overview = []

for year in sorted(lista_com):

    overview.append(
        {
            "year": year,
            "lista_com_rows": len(lista_com[year]),
            "valori_com_rows": len(valori_com[year]),
            "valori_per_rows": len(valori_per[year]),
            "valori_res_rows": len(valori_res[year]),
        }
    )

overview = pd.DataFrame(overview)

overview

,year,lista_com_rows,valori_com_rows,valori_per_rows,valori_res_rows
0,2011,7715,7715,7715,7715
1,2012,7715,7715,7715,7715
2,2013,7698,7698,7698,7698
3,2014,7665,7665,7665,7665
4,2015,7661,7661,7661,7661
5,2016,7645,7645,7645,7645
6,2017,7624,7624,7624,7639
7,2018,7603,7603,7604,7603
8,2019,7564,7564,7564,7564
9,2020,7562,7562,7562,7562


In [94]:
def get_codcom_column(df: pd.DataFrame) -> str:
    """Return the municipality code column."""

    codcom_columns = [
        col
        for col in df.columns
        if "CodCom" in col
    ]

    if len(codcom_columns) != 1:
        raise ValueError(
            f"Expected exactly one CodCom column, "
            f"found: {codcom_columns}"
        )

    return codcom_columns[0]

for year in sorted(lista_com):

    key = get_codcom_column(lista_com[year])

    print(
        f"{year}: {key} | "
        f"unique={lista_com[year][key].nunique():,} | "
        f"duplicates={lista_com[year][key].duplicated().sum():,}"
    )

2011: 2011_CodCom | unique=7,715 | duplicates=0
2012: 2012_CodCom | unique=7,715 | duplicates=0
2013: 2013_CodCom | unique=7,698 | duplicates=0
2014: 2014_CodCom | unique=7,665 | duplicates=0
2015: 2015_CodCom | unique=7,661 | duplicates=0
2016: 2016_CodCom | unique=7,645 | duplicates=0
2017: 2017_CodCom | unique=7,624 | duplicates=0
2018: 2018_CodCom | unique=7,603 | duplicates=0
2019: 2019_CodCom | unique=7,564 | duplicates=0
2020: 2020_CodCom | unique=7,562 | duplicates=0
2021: 2021_CodCom | unique=7,563 | duplicates=0
2022: 2022_CodCom | unique=7,563 | duplicates=0
2023: 2023_CodCom | unique=7,563 | duplicates=0
2024: 2024_CodCom | unique=7,561 | duplicates=0
2025: 2025_CodCom | unique=7,560 | duplicates=0


In [95]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize common geographical column names."""

    df = df.copy()

    rename_map = {}

    for column in df.columns:
        column_upper = column.upper()

        if column_upper == "AREA":
            rename_map[column] = "Area"

        elif column_upper == "REGIONE":
            rename_map[column] = "Regione"

        elif column_upper in {"PROV", "PROVINCIA"}:
            rename_map[column] = "Provincia"

        elif "CODCOM" in column_upper:
            rename_map[column] = "CodCom"

        elif "CODFITT" in column_upper:
            rename_map[column] = "CodCom"

        elif column_upper == "COD_COM":
            rename_map[column] = "CodCom"

    return df.rename(columns=rename_map)

for datasets in [
    lista_com,
    valori_com,
    valori_per,
    valori_res,
]:
    for year in datasets:
        datasets[year] = normalize_columns(
            datasets[year]
        )


In [96]:
for year in sorted(valori_res):
    print(
        year,
        [
            col
            for col in valori_res[year].columns
            if col in ["Area", "Regione", "Provincia", "CodCom"]
        ],
    )

2011 ['Area', 'Regione', 'Provincia', 'CodCom']
2012 ['Area', 'Regione', 'Provincia', 'CodCom']
2013 ['Area', 'Regione', 'Provincia', 'CodCom']
2014 ['Area', 'Regione', 'Provincia', 'CodCom']
2015 ['Area', 'Regione', 'Provincia', 'CodCom']
2016 ['Area', 'Regione', 'Provincia', 'CodCom']
2017 ['Area', 'Regione', 'Provincia', 'CodCom']
2018 ['Area', 'Regione', 'Provincia', 'CodCom']
2019 ['Area', 'Regione', 'Provincia', 'CodCom']
2020 ['Area', 'Regione', 'Provincia', 'CodCom']
2021 ['Area', 'Regione', 'Provincia', 'CodCom']
2022 ['Area', 'Regione', 'Provincia', 'CodCom']
2023 ['Area', 'Regione', 'Provincia', 'CodCom']
2024 ['Area', 'Regione', 'Provincia', 'CodCom']
2025 ['Area', 'Regione', 'Provincia', 'CodCom']


In [97]:
for year in sorted(lista_com):

    print(
        year,
        lista_com[year]["CodCom"].nunique(),
        valori_res[year]["CodCom"].nunique(),
        valori_com[year]["CodCom"].nunique(),
        valori_per[year]["CodCom"].nunique(),
    )

2011 7715 7715 7715 7715
2012 7715 7715 7715 7715
2013 7698 7698 7698 7698
2014 7665 7665 7665 7665
2015 7661 7661 7661 7661
2016 7645 7645 7645 7645
2017 7624 7626 7624 7624
2018 7603 7603 7603 7603
2019 7564 7564 7564 7564
2020 7562 7562 7562 7562
2021 7563 7563 7563 7563
2022 7563 7563 7563 7563
2023 7563 7563 7563 7563
2024 7561 7561 7561 7561
2025 7560 7560 7560 7560


In [98]:
for year in sorted(valori_res):

    print(
        year,
        valori_res[year].columns.tolist()
    )

2011 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2011 fino a 50 mq', 'NTN 2011  50 -| 85 mq', 'NTN 2011  85 -| 115 mq', 'NTN 2011 115 -| 145 mq', 'NTN 2011 oltre 145 mq', 'NTN 2011']
2012 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2012 fino a 50 mq', 'NTN 2012 50 -| 85 mq', 'NTN 2012 85 -| 115 mq', 'NTN 2012 115 -| 145 mq', 'NTN 2012  oltre 145 mq', 'NTN 2012']
2013 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2013 fino a 50 mq', 'NTN 2013 50 -| 85 mq', 'NTN 2013 85 -| 115 mq', 'NTN 2013 115 -| 145 mq', 'NTN 2013 oltre 145 mq', 'NTN 2013']
2014 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2014 fino a 50 mq', 'NTN 2014 50 -| 85 mq', 'NTN 2014 85 -| 115 mq', 'NTN 2014 115 -| 145 mq', 'NTN 2014 oltre 145', 'NTN 2014']
2015 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2015 fino a 50 mq', 'NTN 2015 50 -| 85 mq', 'NTN 2015  85 -| 115 mq', 'NTN 2015 115 -| 145 mq', 'NTN 2015  oltre 145 mq', 'NTN 2015_TOTALE']
2016 ['Area', 'Regione', 'Provincia', 'CodCom', 'NTN 2016 fino a 5

In [99]:
for year in sorted(valori_res):

    duplicates = valori_res[year]["CodCom"].duplicated().sum()

    if duplicates > 0:
        print(
            f"{year}: {duplicates} duplicated CodCom"
        )

2017: 13 duplicated CodCom


In [100]:
for year in sorted(valori_res):

    mask = valori_res[year]["CodCom"].duplicated(
        keep=False
    )

    if mask.any():
        print(f"\n--- {year} ---")
        display(
            valori_res[year].loc[mask].head(20)
        )


--- 2017 ---


,Area,Regione,Provincia,CodCom,NTN 2017 fino a 50 mq,NTN 2017 50 -| 85 mq,NTN 2017 85 -| 115 mq,NTN 2017 115 -| 145 mq,NTN 2017 oltre 145 mq,NTN 2017
7624,Nord Ovest,Liguria,IM,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7625,Nord Ovest,Piemonte,AL,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7626,Centro,Toscana,LI,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7627,Centro,Toscana,AR,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7628,Nord Ovest,Lombardia,CO,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7629,Nord Ovest,Lombardia,LC,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7630,Nord Ovest,Lombardia,MN,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7631,Nord Est,Friuli- Venezia Giulia,UD,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7632,Nord Est,Veneto,PD,non nati,0.0,0.0,0.0,0.0,0.0,0.0
7633,Sud,Calabria,CS,non nati,0.0,0.0,0.0,0.0,0.0,0.0


In [101]:
for year in sorted(valori_res):

    duplicates = (
        valori_res[year]
        .loc[
            valori_res[year]["CodCom"].duplicated(
                keep=False
            )
        ]
        ["CodCom"]
        .value_counts()
    )

    if not duplicates.empty:
        print(f"\n{year}")
        print(duplicates)


2017
CodCom
non nati    10
non nato     5
Name: count, dtype: int64


In [102]:
valori_res_clean = {}

for year, df in valori_res.items():

    valori_res_clean[year] = df.loc[
        ~df["CodCom"].isin(["non nati", "non nato"])
    ].copy()


In [103]:
for name, datasets in {
    "lista_com": lista_com,
    "valori_res": valori_res,
    "valori_com": valori_com,
    "valori_per": valori_per,
}.items():

    print(f"\n{name}")

    for year, df in sorted(datasets.items()):

        duplicates = df["CodCom"].duplicated().sum()

        if duplicates > 0:
            print(
                f"{year}: "
                f"{duplicates} duplicated CodCom"
            )


lista_com

valori_res
2017: 13 duplicated CodCom

valori_com

valori_per


In [104]:
transactions = {}

for year in sorted(lista_com):

    transactions[year] = (
        lista_com[year]
        .merge(
            valori_res_clean[year],
            on="CodCom",
            how="left",
            suffixes=("", "_res"),
            validate="one_to_one",
        )
        .merge(
            valori_com[year],
            on="CodCom",
            how="left",
            suffixes=("", "_com"),
            validate="one_to_one",
        )
        .merge(
            valori_per[year],
            on="CodCom",
            how="left",
            suffixes=("", "_per"),
            validate="one_to_one",
        )
    )

for year, df in transactions.items():

    assert len(df) == len(lista_com[year])

    assert df["CodCom"].duplicated().sum() == 0

print("Merge completed successfully.")

Merge completed successfully.


In [105]:
def normalize_residential_columns(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Normalize residential NTN column names."""

    df = df.copy()

    rename_map = {}

    for column in df.columns:

        column_clean = (
            column
            .lower()
            .replace(" ", "")
            .replace("_", "")
        )

        if "finoa50mq" in column_clean:
            rename_map[column] = "NTN_0_50"

        elif "50-|85mq" in column_clean:
            rename_map[column] = "NTN_50_85"

        elif "85-|115mq" in column_clean:
            rename_map[column] = "NTN_85_115"

        elif "115-|145mq" in column_clean:
            rename_map[column] = "NTN_115_145"

        elif "oltre145mq" in column_clean:
            rename_map[column] = "NTN_145_PLUS"

        elif column_clean in {
            "ntn",
            "ntn2011",
            "ntn2012",
            "ntn2013",
            "ntn2014",
            "ntn2015totale",
            "ntn2016totale",
            "ntn2017",
            "ntn2018",
            "ntn2019",
            "ntn2020",
            "ntn2021",
            "ntn2022",
            "ntn2023",
            "ntn2024",
            "ntn2025",
        }:
            rename_map[column] = "NTN"

    return df.rename(columns=rename_map)

for year in valori_res:

    valori_res[year] = normalize_residential_columns(
        valori_res[year]
    )

In [106]:
for year in transactions:

    transactions[year]["reference_date"] = (
        pd.Timestamp(year=year, month=12, day=31)
    )

In [107]:
transactions[2025].columns.tolist()

['Area',
 'Regione',
 'Provincia',
 'CodCom',
 'Comune',
 'Cap',
 'TAGLIA MERCATO',
 'Area_res',
 'Regione_res',
 'Provincia_res',
 'NTN 2025 fino a 50 mq',
 'NTN 2025 50 -| 85 mq',
 'NTN 2025 85 -| 115 mq',
 'NTN 2025 115 -| 145 mq',
 'NTN 2025 oltre 145 mq',
 'NTN_2025',
 'Area_com',
 'Regione_com',
 'Provincia_com',
 'NTN_2025_Uffici',
 'NTN_2025_Negozi_Lab',
 'NTN_2025_Depositi_Comm_Autorimesse',
 'NTN_2025_TCO_B04',
 'NTN_2025_TCO_D02',
 'NTN_2025_TCO_D05',
 'NTN_2025_TCO_D08',
 'NTN_2025_PRO',
 'NTN_2025_AGR',
 'Area_per',
 'Regione_per',
 'Provincia_per',
 'NTN_2025_Depositi_Pert',
 'NTN_2025_Box',
 'reference_date']

In [108]:
transactions[2011].columns.tolist()

['Area',
 'Regione',
 'Provincia',
 'CodCom',
 'Comune',
 'Area_res',
 'Regione_res',
 'Provincia_res',
 'NTN 2011 fino a 50 mq',
 'NTN 2011  50 -| 85 mq',
 'NTN 2011  85 -| 115 mq',
 'NTN 2011 115 -| 145 mq',
 'NTN 2011 oltre 145 mq',
 'NTN 2011',
 'Area_com',
 'Regione_com',
 'Provincia_com',
 'NTN_2011_Uffici',
 'NTN_2011_Negozi_Lab',
 'NTN_2011_Depositi_Comm',
 'NTN_2011_TCO_B04',
 'NTN_2011_TCO_D02',
 'NTN_2011_TCO_D05',
 'NTN_2011_TCO_D08',
 'NTN_2011_PRO',
 'NTN_2011_AGR',
 'Area_per',
 'Regione_per',
 'Provincia_per',
 'NTN_2011_Box',
 'NTN_2011_Depositi_Pert',
 'reference_date']

In [109]:
for year in sorted(transactions):

    duplicated_columns = [
        col
        for col in transactions[year].columns
        if col.endswith(("_com", "_per", "_res"))
    ]

    if duplicated_columns:
        print(f"\n{year}:")
        print(duplicated_columns)


2011:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2012:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2013:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2014:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2015:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2016:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2017:
['Area_res', 'Regione_res', 'Provincia_res', 'Area_com', 'Regione_com', 'Provincia_com', 'Area_per', 'Regione_per', 'Provincia_per']

2018:
['Area_res', 

In [110]:
def clean_transaction_columns(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Remove duplicated geographical columns after merge."""

    df = df.copy()

    columns_to_drop = [
        "Area_res",
        "Regione_res",
        "Provincia_res",
        "Area_com",
        "Regione_com",
        "Provincia_com",
        "Area_per",
        "Regione_per",
        "Provincia_per",
    ]

    df = df.drop(
        columns=columns_to_drop,
        errors="ignore",
    )

    return df

for year in transactions:

    transactions[year] = clean_transaction_columns(
        transactions[year]
    )

for year in transactions:

    transactions[year] = clean_transaction_columns(
        transactions[year]
    )

In [111]:
transactions[2025].columns.tolist()

['Area',
 'Regione',
 'Provincia',
 'CodCom',
 'Comune',
 'Cap',
 'TAGLIA MERCATO',
 'NTN 2025 fino a 50 mq',
 'NTN 2025 50 -| 85 mq',
 'NTN 2025 85 -| 115 mq',
 'NTN 2025 115 -| 145 mq',
 'NTN 2025 oltre 145 mq',
 'NTN_2025',
 'NTN_2025_Uffici',
 'NTN_2025_Negozi_Lab',
 'NTN_2025_Depositi_Comm_Autorimesse',
 'NTN_2025_TCO_B04',
 'NTN_2025_TCO_D02',
 'NTN_2025_TCO_D05',
 'NTN_2025_TCO_D08',
 'NTN_2025_PRO',
 'NTN_2025_AGR',
 'NTN_2025_Depositi_Pert',
 'NTN_2025_Box',
 'reference_date']

In [112]:
for year in sorted(transactions):

    ntn_columns = [
        col
        for col in transactions[year].columns
        if "NTN" in col.upper()
    ]

    print(f"\n{year}:")
    print(ntn_columns)


2011:
['NTN 2011 fino a 50 mq', 'NTN 2011  50 -| 85 mq', 'NTN 2011  85 -| 115 mq', 'NTN 2011 115 -| 145 mq', 'NTN 2011 oltre 145 mq', 'NTN 2011', 'NTN_2011_Uffici', 'NTN_2011_Negozi_Lab', 'NTN_2011_Depositi_Comm', 'NTN_2011_TCO_B04', 'NTN_2011_TCO_D02', 'NTN_2011_TCO_D05', 'NTN_2011_TCO_D08', 'NTN_2011_PRO', 'NTN_2011_AGR', 'NTN_2011_Box', 'NTN_2011_Depositi_Pert']

2012:
['NTN 2012 fino a 50 mq', 'NTN 2012 50 -| 85 mq', 'NTN 2012 85 -| 115 mq', 'NTN 2012 115 -| 145 mq', 'NTN 2012  oltre 145 mq', 'NTN 2012', 'NTN_2012_Uffici', 'NTN_2012_Negozi_Lab', 'NTN_2012_Depositi_Comm', 'NTN_2012_TCO_B04', 'NTN_2012_TCO_D02', 'NTN_2012_TCO_D05', 'NTN_2012_TCO_D08', 'NTN_2012_PRO', 'NTN_2012_AGR', 'NTN_2012_Box', 'NTN_2012_Depositi_Pert']

2013:
['NTN 2013 fino a 50 mq', 'NTN 2013 50 -| 85 mq', 'NTN 2013 85 -| 115 mq', 'NTN 2013 115 -| 145 mq', 'NTN 2013 oltre 145 mq', 'NTN 2013', 'NTN_2013_Uffici', 'NTN_2013_Negozi_Lab', 'NTN_2013_Depositi_Comm', 'NTN_2013_TCO_B04', 'NTN_2013_TCO_D02', 'NTN_2013_

In [113]:
def normalize_transaction_columns(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Normalize NTN column names across years."""

    df = df.copy()

    rename_map = {}

    for column in df.columns:

        col = (
            column
            .strip()
            .lower()
            .replace(" ", "")
        )

        # Remove year from column name
        col = re.sub(r"ntn[_]?\d{4}", "ntn", col)

        # ---------------------------------------------------------
        # Residential NTN by surface class
        # ---------------------------------------------------------

        if "finoa50mq" in col or "finoa50_mq" in col:
            rename_map[column] = "NTN_0_50"

        elif "50-|85mq" in col:
            rename_map[column] = "NTN_50_85"

        elif "85-|115mq" in col:
            rename_map[column] = "NTN_85_115"

        elif "115-|145mq" in col:
            rename_map[column] = "NTN_115_145"

        elif "oltre145" in col:
            rename_map[column] = "NTN_145_PLUS"

        # ---------------------------------------------------------
        # Total residential NTN
        # ---------------------------------------------------------

        elif col in {"ntn", "ntntotale"}:
            rename_map[column] = "NTN"

        # ---------------------------------------------------------
        # Other property categories
        # ---------------------------------------------------------

        elif "uffici" in col:
            rename_map[column] = "NTN_UFFICI"

        elif "negozi_lab" in col or "negozilab" in col:
            rename_map[column] = "NTN_NEGOZI_LAB"

        elif "depositi_comm" in col:
            rename_map[column] = "NTN_DEPOSITI_COMM"

        elif "tco_b04" in col:
            rename_map[column] = "NTN_TCO_B04"

        elif "tco_d02" in col:
            rename_map[column] = "NTN_TCO_D02"

        elif "tco_d05" in col:
            rename_map[column] = "NTN_TCO_D05"

        elif "tco_d08" in col:
            rename_map[column] = "NTN_TCO_D08"

        elif col.endswith("pro"):
            rename_map[column] = "NTN_PRO"

        elif col.endswith("agr"):
            rename_map[column] = "NTN_AGR"

        elif "box" in col:
            rename_map[column] = "NTN_BOX"

        elif "depositi_pert" in col:
            rename_map[column] = "NTN_DEPOSITI_PERT"

    return df.rename(columns=rename_map)

for year in transactions:

    transactions[year] = normalize_transaction_columns(
        transactions[year]
    )

for year in sorted(transactions):

    ntn_columns = [
        col
        for col in transactions[year].columns
        if col.startswith("NTN")
    ]

    print(f"\n{year}:")
    print(ntn_columns)


2011:
['NTN_0_50', 'NTN_50_85', 'NTN_85_115', 'NTN_115_145', 'NTN_145_PLUS', 'NTN', 'NTN_UFFICI', 'NTN_NEGOZI_LAB', 'NTN_DEPOSITI_COMM', 'NTN_TCO_B04', 'NTN_TCO_D02', 'NTN_TCO_D05', 'NTN_TCO_D08', 'NTN_PRO', 'NTN_AGR', 'NTN_BOX', 'NTN_DEPOSITI_PERT']

2012:
['NTN_0_50', 'NTN_50_85', 'NTN_85_115', 'NTN_115_145', 'NTN_145_PLUS', 'NTN', 'NTN_UFFICI', 'NTN_NEGOZI_LAB', 'NTN_DEPOSITI_COMM', 'NTN_TCO_B04', 'NTN_TCO_D02', 'NTN_TCO_D05', 'NTN_TCO_D08', 'NTN_PRO', 'NTN_AGR', 'NTN_BOX', 'NTN_DEPOSITI_PERT']

2013:
['NTN_0_50', 'NTN_50_85', 'NTN_85_115', 'NTN_115_145', 'NTN_145_PLUS', 'NTN', 'NTN_UFFICI', 'NTN_NEGOZI_LAB', 'NTN_DEPOSITI_COMM', 'NTN_TCO_B04', 'NTN_TCO_D02', 'NTN_TCO_D05', 'NTN_TCO_D08', 'NTN_PRO', 'NTN_AGR', 'NTN_BOX', 'NTN_DEPOSITI_PERT']

2014:
['NTN_0_50', 'NTN_50_85', 'NTN_85_115', 'NTN_115_145', 'NTN_145_PLUS', 'NTN', 'NTN_UFFICI', 'NTN_NEGOZI_LAB', 'NTN_DEPOSITI_COMM', 'NTN_TCO_B04', 'NTN_TCO_D02', 'NTN_TCO_D05', 'NTN_TCO_D08', 'NTN_PRO', 'NTN_AGR', 'NTN_BOX', 'NTN_DEPOSITI

In [114]:
expected_ntn_columns = {
    "NTN_0_50",
    "NTN_50_85",
    "NTN_85_115",
    "NTN_115_145",
    "NTN_145_PLUS",
    "NTN",
    "NTN_UFFICI",
    "NTN_NEGOZI_LAB",
    "NTN_DEPOSITI_COMM",
    "NTN_TCO_B04",
    "NTN_TCO_D02",
    "NTN_TCO_D05",
    "NTN_TCO_D08",
    "NTN_PRO",
    "NTN_AGR",
    "NTN_BOX",
    "NTN_DEPOSITI_PERT",
}

for year, df in sorted(transactions.items()):

    actual = {
        col
        for col in df.columns
        if col.startswith("NTN")
    }

    missing = expected_ntn_columns - actual
    extra = actual - expected_ntn_columns

    if missing or extra:
        print(f"\n{year}")
        print("Missing:", missing)
        print("Extra:", extra)



2015
Missing: {'NTN'}
Extra: {'NTN 2015_TOTALE'}

2016
Missing: {'NTN'}
Extra: {'NTN 2016_TOTALE'}


In [115]:
for year, df in transactions.items():

    duplicated = df.columns[
        df.columns.duplicated()
    ]

    if len(duplicated) > 0:
        print(
            year,
            duplicated.tolist()
        )

In [116]:
for year, df in transactions.items():

    print(
        f"{year}: "
        f"rows={len(df):,} | "
        f"unique CodCom={df['CodCom'].nunique():,} | "
        f"duplicates={df['CodCom'].duplicated().sum():,}"
    )

2011: rows=7,715 | unique CodCom=7,715 | duplicates=0
2012: rows=7,715 | unique CodCom=7,715 | duplicates=0
2013: rows=7,698 | unique CodCom=7,698 | duplicates=0
2014: rows=7,665 | unique CodCom=7,665 | duplicates=0
2015: rows=7,661 | unique CodCom=7,661 | duplicates=0
2016: rows=7,645 | unique CodCom=7,645 | duplicates=0
2017: rows=7,624 | unique CodCom=7,624 | duplicates=0
2018: rows=7,603 | unique CodCom=7,603 | duplicates=0
2019: rows=7,564 | unique CodCom=7,564 | duplicates=0
2020: rows=7,562 | unique CodCom=7,562 | duplicates=0
2021: rows=7,563 | unique CodCom=7,563 | duplicates=0
2022: rows=7,563 | unique CodCom=7,563 | duplicates=0
2023: rows=7,563 | unique CodCom=7,563 | duplicates=0
2024: rows=7,561 | unique CodCom=7,561 | duplicates=0
2025: rows=7,560 | unique CodCom=7,560 | duplicates=0


In [117]:
transactions_df = pd.concat(
    transactions.values(),
    ignore_index=True,
)

In [118]:
transactions_df.shape

(114262, 31)

In [119]:
transactions_df.head()

,Area,Regione,Provincia,CodCom,Comune,NTN_0_50,NTN_50_85,NTN_85_115,NTN_115_145,NTN_145_PLUS,...,NTN_DEPOSITI_PERT,reference_date,NTN 2015_TOTALE,NTN 2016_TOTALE,COD_ISTAT,Unnamed: 13,Unnamed: 14,Cod_Istat,Cap,TAGLIA MERCATO
0,Nord Ovest,Liguria,GE,A388,ARENZANO,21.83,78.94,41.78,21.37,11.00,...,28.19,2011-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Nord Ovest,Liguria,GE,A506,AVEGNO,0.00,8.00,11.51,7.50,3.00,...,13.76,2011-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nord Ovest,Liguria,GE,A658,BARGAGLI,1.00,20.50,16.00,9.00,6.75,...,11.50,2011-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nord Ovest,Liguria,GE,A922,BOGLIASCO,2.00,19.73,17.13,2.17,5.00,...,11.17,2011-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Nord Ovest,Liguria,GE,B067,BORZONASCA,3.00,6.52,5.75,5.50,5.50,...,5.50,2011-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [120]:
transactions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114262 entries, 0 to 114261
Data columns (total 31 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Area               114262 non-null  str           
 1   Regione            114262 non-null  str           
 2   Provincia          112882 non-null  str           
 3   CodCom             114262 non-null  str           
 4   Comune             114262 non-null  str           
 5   NTN_0_50           114262 non-null  float64       
 6   NTN_50_85          114262 non-null  float64       
 7   NTN_85_115         114262 non-null  float64       
 8   NTN_115_145        114262 non-null  float64       
 9   NTN_145_PLUS       114262 non-null  float64       
 10  NTN                98956 non-null   float64       
 11  NTN_UFFICI         114262 non-null  float64       
 12  NTN_NEGOZI_LAB     114262 non-null  float64       
 13  NTN_DEPOSITI_COMM  114262 non-null  float64       
 14 

In [121]:
transactions_df["reference_date"].min(), transactions_df["reference_date"].max()

(Timestamp('2011-12-31 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [122]:
transactions_df["reference_date"].dt.year.value_counts().sort_index()

reference_date
2011    7715
2012    7715
2013    7698
2014    7665
2015    7661
2016    7645
2017    7624
2018    7603
2019    7564
2020    7562
2021    7563
2022    7563
2023    7563
2024    7561
2025    7560
Name: count, dtype: int64

In [123]:
transactions_df.duplicated(
    subset=["CodCom", "reference_date"]
).sum()

np.int64(0)

In [124]:
transactions_df["NTN"].value_counts().head(20)

NTN
2.0     2190
3.0     2131
4.0     2048
1.0     2034
5.0     1736
6.0     1618
0.0     1427
7.0     1418
8.0     1179
9.0     1000
10.0     934
11.0     857
12.0     746
13.0     669
14.0     592
15.0     511
4.5      487
16.0     467
6.5      462
8.5      459
Name: count, dtype: int64